In [1]:
import pandas as pd
import numpy as np

In [2]:
aq = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Raw data/Air Quality/Air_Quality_20251114.csv')

In [3]:
print("Shape:", aq.shape)
print("Columns:", aq.columns.tolist())
print("First 5 rows:\n", aq.head())
print("Dtypes:\n", aq.dtypes)

Shape: (18862, 12)
Columns: ['Unique ID', 'Indicator ID', 'Name', 'Measure', 'Measure Info', 'Geo Type Name', 'Geo Join ID', 'Geo Place Name', 'Time Period', 'Start_Date', 'Data Value', 'Message']
First 5 rows:
    Unique ID  Indicator ID                     Name Measure Measure Info  \
0     878218           386               Ozone (O3)    Mean          ppb   
1     876975           375   Nitrogen dioxide (NO2)    Mean          ppb   
2     876900           375   Nitrogen dioxide (NO2)    Mean          ppb   
3     877140           375   Nitrogen dioxide (NO2)    Mean          ppb   
4     874556           365  Fine particles (PM 2.5)    Mean       mcg/m3   

  Geo Type Name  Geo Join ID                        Geo Place Name  \
0         UHF42          402                           West Queens   
1         UHF42          501                         Port Richmond   
2         UHF42          207              East Flatbush - Flatbush   
3            CD          205  Fordham and Universit

In [4]:
print("Sample unique place names:", aq['Geo Place Name'].unique()[:20])
print("Total unique Geo Place Names:", aq['Geo Place Name'].nunique())
print("Geo Type counts:\n", aq['Geo Type Name'].value_counts())


Sample unique place names: ['West Queens' 'Port Richmond' 'East Flatbush - Flatbush'
 'Fordham and University Heights (CD5)' 'Rockaways' 'Canarsie - Flatlands'
 'Rockaway and Broad Channel (CD14)' 'Chelsea-Village' 'Greenpoint'
 'Fordham - Bronx Pk' 'Upper East Side-Gramercy'
 'Hunts Point - Mott Haven' 'Bronx' 'Queens' 'Jamaica'
 'Flushing and Whitestone (CD7)' 'East Harlem'
 'Bay Ridge and Dyker Heights (CD10)' 'Riverdale and Fieldston (CD8)'
 'Southeast Queens']
Total unique Geo Place Names: 114
Geo Type counts:
 Geo Type Name
UHF42       7392
CD          6844
UHF34       3570
Borough      880
Citywide     176
Name: count, dtype: int64


In [5]:
# For explicit borough-level records
borough_rows = aq[aq['Geo Type Name'] == 'Borough']
print("Distinct boroughs in 'Geo Place Name':", borough_rows['Geo Place Name'].unique())
print("Borough record counts:\n", borough_rows['Geo Place Name'].value_counts())

# For all records: count by borough keyword in place name
BOROUGHS = ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
for borough in BOROUGHS:
    print(f"Records with '{borough}' in Geo Place Name:", aq['Geo Place Name'].str.contains(borough, case=False).sum())


Distinct boroughs in 'Geo Place Name': ['Bronx' 'Queens' 'Staten Island' 'Brooklyn' 'Manhattan']
Borough record counts:
 Geo Place Name
Bronx            176
Queens           176
Staten Island    176
Brooklyn         176
Manhattan        176
Name: count, dtype: int64
Records with 'Bronx' in Geo Place Name: 959
Records with 'Brooklyn' in Geo Place Name: 292
Records with 'Manhattan' in Geo Place Name: 457
Records with 'Queens' in Geo Place Name: 1135
Records with 'Staten Island' in Geo Place Name: 176


In [6]:
# Ensure Start_Date is converted to datetime
aq['Start_Date'] = pd.to_datetime(aq['Start_Date'], errors='coerce')
BOROUGHS = ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']

for boro in BOROUGHS:
    # For strict 'Borough' Geo Type (borough average)
    years_boro = aq[(aq['Geo Type Name'] == 'Borough') & (aq['Geo Place Name'] == boro)]['Start_Date'].dt.year.dropna()
    if not years_boro.empty:
        print(f"{boro} (Borough records): {int(years_boro.min())}–{int(years_boro.max())} ({years_boro.nunique()} years)")
    # Also include records containing the borough in place name if needed
    years_any = aq[aq['Geo Place Name'].str.contains(boro, case=False, na=False)]['Start_Date'].dt.year.dropna()
    if not years_any.empty:
        print(f"{boro} (ANY records): {int(years_any.min())}–{int(years_any.max())} ({years_any.nunique()} years)")

Bronx (Borough records): 2005–2023 (17 years)
Bronx (ANY records): 2005–2023 (17 years)
Brooklyn (Borough records): 2005–2023 (17 years)
Brooklyn (ANY records): 2005–2023 (17 years)
Manhattan (Borough records): 2005–2023 (17 years)
Manhattan (ANY records): 2005–2023 (17 years)
Queens (Borough records): 2005–2023 (17 years)
Queens (ANY records): 2005–2023 (17 years)
Staten Island (Borough records): 2005–2023 (17 years)
Staten Island (ANY records): 2005–2023 (17 years)


In [7]:
# Aggregate to get annual borough means (or use median, min, max, etc., as needed)
borough_year = aq[aq['Geo Type Name'] == 'Borough'].copy()
borough_year['Year'] = borough_year['Start_Date'].dt.year
borough_avg = borough_year.groupby(['Geo Place Name', 'Year'])['Data Value'].mean().reset_index()
borough_avg.rename(columns={'Geo Place Name': 'Borough', 'Data Value': 'MeanValue'}, inplace=True)

print(borough_avg.head())

  Borough  Year  MeanValue
0   Bronx  2005  56.936318
1   Bronx  2008  19.735000
2   Bronx  2009  40.952941
3   Bronx  2010  25.847810
4   Bronx  2011  14.435556


In [8]:
print("Any missing borough-year pairs?\n", borough_avg.isna().sum())
print("\nCheck for duplicates (should be 0):", borough_avg.duplicated(subset=['Borough', 'Year']).sum())
print("\nQuick completeness check (counts each year per borough):\n", borough_avg.groupby('Borough')['Year'].nunique())

Any missing borough-year pairs?
 Borough      0
Year         0
MeanValue    0
dtype: int64

Check for duplicates (should be 0): 0

Quick completeness check (counts each year per borough):
 Borough
Bronx            17
Brooklyn         17
Manhattan        17
Queens           17
Staten Island    17
Name: Year, dtype: int64


In [9]:
cols_to_drop = ['Unique ID', 'Indicator ID', 'Measure', 'Message']
aq_clean = aq.drop(columns=cols_to_drop)

In [10]:
kept_col_order = ['Geo Join ID', 'Geo Place Name', 'Geo Type Name', 'Start_Date', 'Time Period', 'Name', 'Measure Info', 'Data Value']
aq_clean = aq_clean[kept_col_order]

In [11]:
# Sort by 'Time Period' (season/year or period) and then by 'Geo Place Name' for location grouping
aq_clean = aq_clean.sort_values(['Time Period', 'Geo Place Name']).reset_index(drop=True)
print(aq_clean.head(10))

   Geo Join ID                      Geo Place Name Geo Type Name Start_Date  \
0          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
1          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
2          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
3          404               Bayside - Little Neck         UHF42 2005-01-01   
4          404               Bayside - Little Neck         UHF42 2005-01-01   
5          404               Bayside - Little Neck         UHF42 2005-01-01   
6          404               Bayside - Little Neck         UHF42 2005-01-01   
7          404               Bayside - Little Neck         UHF42 2005-01-01   
8          411      Bayside and Little Neck (CD11)            CD 2005-01-01   
9          411      Bayside and Little Neck (CD11)            CD 2005-01-01   

  Time Period                                    Name     Measure Info  \
0        2005           Annual vehicle miles traveled  p

In [12]:
print("Final shape:", aq_clean.shape)
print("Final columns:", aq_clean.columns.tolist())
print("First 5 rows:\n", aq_clean.head())

print("Missing values by column:\n", aq_clean.isna().sum())

# Check for full duplicates
print("Full duplicated rows:", aq_clean.duplicated().sum())
# Check for key-duplicates (by Time Period, Place Name, Name, Data Value if those are your analytical keys)
dup_keys = ['Time Period', 'Geo Place Name', 'Name', 'Data Value']
print("Duplicated rows by keys:", aq_clean.duplicated(subset=dup_keys).sum())

print("Unique Time Periods:", aq_clean['Time Period'].unique())
print("Unique Geo Place Names:", aq_clean['Geo Place Name'].nunique())
print("Date range:", aq_clean['Start_Date'].min(), "to", aq_clean['Start_Date'].max())
print("Data Value range:", aq_clean['Data Value'].min(), "to", aq_clean['Data Value'].max())
print("Unique pollutant types (Name):", aq_clean['Name'].unique())

Final shape: (18862, 8)
Final columns: ['Geo Join ID', 'Geo Place Name', 'Geo Type Name', 'Start_Date', 'Time Period', 'Name', 'Measure Info', 'Data Value']
First 5 rows:
    Geo Join ID                      Geo Place Name Geo Type Name Start_Date  \
0          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
1          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
2          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
3          404               Bayside - Little Neck         UHF42 2005-01-01   
4          404               Bayside - Little Neck         UHF42 2005-01-01   

  Time Period                                    Name     Measure Info  \
0        2005           Annual vehicle miles traveled  per square mile   
1        2005    Annual vehicle miles traveled (cars)  per square mile   
2        2005  Annual vehicle miles traveled (trucks)  per square mile   
3        2005           Annual vehicle miles traveled  pe

In [13]:
print(aq_clean.head(10))
print(aq_clean.tail(10))

   Geo Join ID                      Geo Place Name Geo Type Name Start_Date  \
0          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
1          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
2          310  Bay Ridge and Dyker Heights (CD10)            CD 2005-01-01   
3          404               Bayside - Little Neck         UHF42 2005-01-01   
4          404               Bayside - Little Neck         UHF42 2005-01-01   
5          404               Bayside - Little Neck         UHF42 2005-01-01   
6          404               Bayside - Little Neck         UHF42 2005-01-01   
7          404               Bayside - Little Neck         UHF42 2005-01-01   
8          411      Bayside and Little Neck (CD11)            CD 2005-01-01   
9          411      Bayside and Little Neck (CD11)            CD 2005-01-01   

  Time Period                                    Name     Measure Info  \
0        2005           Annual vehicle miles traveled  p

In [16]:
aq_clean.to_csv("/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/nyc_air_clean_by_period_place.csv", index=False)
print("Saved final cleaned file: nyc_air_clean_by_period_place.csv")

Saved final cleaned file: nyc_air_clean_by_period_place.csv
